# 03 — Watson exact duplicate check

Downloads [SNLI](https://nlp.stanford.edu/projects/snli/), [MNLI](https://cims.nyu.edu/~sbowman/multinli/), [ANLI](https://github.com/facebookresearch/anli), and [XNLI](https://github.com/facebookresearch/XNLI), then checks whether their premise–hypothesis pairs appear exactly in Watson.

An exact match requires:
- both sentences, premise and hypothesis, in the same order
- the sentences are taken exactly as is with no changes made (like lowercasing, for example).

Downloads are cached under ignored `data/nli_overlap/`; the first run is large (several hundred MB).

## Saved result — 2026-09-23

All four datasets were obtained. **All 17,315 Watson examples have an exact premise–hypothesis match:**
- 9,815 in MNLI
- 7,500 in XNLI.
- SNLI and ANLI have zero matches in the scanned files.

In [1]:
from pathlib import Path
from collections import defaultdict
import json
import subprocess
import zipfile

import pandas as pd
from IPython.display import display

# Locate the repo from either its root or notebooks directory.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "data/raw/train.csv").exists(), "Run from the repo or notebooks directory."
CACHE = ROOT / "data/nli_overlap"
CACHE.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 160)

# Reset results on each run.
acquisition = []
coverage = []
matches = []

## Load all Watson examples

In [2]:
# Keep file and row IDs for later inspection; labels are not needed.
watson_parts = []
for filename in ["train.csv", "test.csv"]:
    frame = pd.read_csv(ROOT / "data/raw" / filename, keep_default_na=False)
    assert {"id", "premise", "hypothesis"}.issubset(frame.columns)
    assert frame[["premise", "hypothesis"]].map(lambda x: isinstance(x, str) and bool(x.strip())).all().all()
    frame["watson_file"] = filename
    watson_parts.append(frame[["id", "premise", "hypothesis", "watson_file"]])
watson = pd.concat(watson_parts, ignore_index=True)

# Index complete pairs, retaining repeated Watson rows.
watson_index = defaultdict(list)
for row in watson.to_dict("records"):
    watson_index[(row["premise"], row["hypothesis"])].append(row)
print(f"Watson: {len(watson):,} rows; {len(watson_index):,} distinct sentence pairs.")
display(watson.groupby("watson_file").size().rename("rows").to_frame())

Watson: 17,315 rows; 17,315 distinct sentence pairs.


,rows
watson_file,
test.csv,5195
train.csv,12120


The cells above show the total number of examples found in Watson's train and test sets. Now we will compare these examples to those found in the 4 NLI Datasets mentioned above this.

## Download and compare each dataset

Each dataset runs independently. A failed download is reported as unavailable.

In [3]:
def fetch_archive(url, filename):
    # Reuse complete downloads; replace only after a successful transfer.
    path = CACHE / filename
    if not path.exists():
        temporary = path.with_suffix(".part")
        print(f"Downloading {filename} ...", flush=True)
        try:
            # Use curl's system certificate trust; keep HTTPS verification enabled.
            subprocess.run(
                ["curl", "--fail", "--location", "--silent", "--show-error",
                 "--retry", "2", "--connect-timeout", "30", "--max-time", "600",
                 "--output", str(temporary), url],
                check=True, capture_output=True, text=True,
            )
            if not zipfile.is_zipfile(temporary):
                raise ValueError("Response is not a ZIP archive")
            temporary.replace(path)
        finally:
            temporary.unlink(missing_ok=True)
    return path


def scan_dataset(name, url, filename):
    # Replace this dataset's prior results if its cell is rerun.
    for results in [acquisition, coverage, matches]:
        results[:] = [r for r in results if r["dataset"] != name]
    local_coverage, local_matches = [], []
    try:
        path = fetch_archive(url, filename)
        with zipfile.ZipFile(path) as archive:
            members = [m for m in archive.namelist() if m.endswith(".jsonl") and not m.startswith("__MACOSX/")]
            if not members:
                raise ValueError("No JSONL dataset files found")
            for member in sorted(members):
                count = 0
                with archive.open(member) as stream:
                    for line in stream:
                        if not line.strip():
                            continue
                        row = json.loads(line)
                        premise = row.get("sentence1", row.get("context", row.get("premise")))
                        hypothesis = row.get("sentence2", row.get("hypothesis"))
                        if not isinstance(premise, str) or not isinstance(hypothesis, str):
                            raise ValueError(f"Missing sentence text in {member}, row {count + 1}")
                        count += 1
                        for found in watson_index.get((premise, hypothesis), []):
                            local_matches.append({
                                "dataset": name, "source_file": member,
                                "source_row": count,
                                "source_id": row.get("pairID", row.get("uid", "")),
                                "language": row.get("language", "en"),
                                "watson_file": found["watson_file"], "watson_id": found["id"],
                                "premise": premise, "hypothesis": hypothesis,
                            })
                if count == 0:
                    raise ValueError(f"Empty dataset file: {member}")
                local_coverage.append({"dataset": name, "source_file": member, "rows": count})
        coverage.extend(local_coverage)
        matches.extend(local_matches)
        acquisition.append({"dataset": name, "status": "loaded", "url": url, "error": ""})
        distinct = len({(r["watson_file"], r["watson_id"]) for r in local_matches})
        print(f"{name}: scanned {sum(r['rows'] for r in local_coverage):,} rows; {distinct:,} Watson rows matched.")
    except Exception as error:
        acquisition.append({"dataset": name, "status": "unavailable", "url": url, "error": str(error)})
        print(f"{name}: unavailable — {type(error).__name__}: {error}")

In [4]:
# Obtain and scan SNLI independently.
scan_dataset('SNLI', 'https://nlp.stanford.edu/projects/snli/snli_1.0.zip', 'snli_1.0.zip')

SNLI: scanned 570,152 rows; 0 Watson rows matched.


In [5]:
# Obtain and scan MNLI independently.
scan_dataset('MNLI', 'https://cims.nyu.edu/~sbowman/multinli/multinli_1.0.zip', 'multinli_1.0.zip')

MNLI: scanned 412,702 rows; 9,815 Watson rows matched.


In [6]:
# Obtain and scan ANLI independently.
scan_dataset('ANLI', 'https://dl.fbaipublicfiles.com/anli/anli_v1.0.zip', 'anli_v1.0.zip')

ANLI: scanned 169,265 rows; 0 Watson rows matched.


In [7]:
# Obtain and scan XNLI independently.
scan_dataset('XNLI', 'https://dl.fbaipublicfiles.com/XNLI/XNLI-1.0.zip', 'XNLI-1.0.zip')

XNLI: scanned 112,500 rows; 7,500 Watson rows matched.


## Make sure to count each Watson example only once

We don't want to mistakenly think that all of the Watson examples are duplicates of something in the four NLI datasets. For example, if one Watson example was found in multiple NLI datasets, it may have counted as more than one in the tallies above this. So check that each of the Watson examples is indeed a duplicate of a pair found in the NLI datasets.

In [8]:
# Count each Watson row once per dataset, even if the source repeats it.
summary_rows = []
for name in ["SNLI", "MNLI", "ANLI", "XNLI"]:
    state = next((r["status"] for r in acquisition if r["dataset"] == name), "not run")
    found = [r for r in matches if r["dataset"] == name]
    summary_rows.append({
        "dataset": name, "status": state,
        "source_rows_scanned": sum(r["rows"] for r in coverage if r["dataset"] == name) if state == "loaded" else None,
        "Watson_rows_matched": len({(r["watson_file"], r["watson_id"]) for r in found}) if state == "loaded" else None,
        "distinct_pairs_matched": len({(r["premise"], r["hypothesis"]) for r in found}) if state == "loaded" else None,
    })
display(pd.DataFrame(summary_rows))
all_matched = {(r["watson_file"], r["watson_id"]) for r in matches}
if all_matched:
    print(f"YES: {len(all_matched):,} of {len(watson):,} Watson rows have strict duplicates in the scanned data.")
elif all(r["status"] == "loaded" for r in summary_rows):
    print("No strict duplicates found in the scanned files. Translated duplicates have not been checked.")
else:
    print("Incomplete: no strict duplicates found so far, but some datasets were not scanned.")

# Show actual coverage rather than assuming every release includes all files.
display(pd.DataFrame(coverage))
failures = [r for r in acquisition if r["status"] != "loaded"]
if failures:
    display(pd.DataFrame(failures))

YES: 17,315 of 17,315 Watson rows have strict duplicates in the scanned data.


,dataset,status,source_rows_scanned,Watson_rows_matched,distinct_pairs_matched
0,SNLI,loaded,570152,0,0
1,MNLI,loaded,412702,9815,9815
2,ANLI,loaded,169265,0,0
3,XNLI,loaded,112500,7500,7500


,dataset,source_file,rows
0,SNLI,snli_1.0/snli_1.0_dev.jsonl,10000
1,SNLI,snli_1.0/snli_1.0_test.jsonl,10000
2,SNLI,snli_1.0/snli_1.0_train.jsonl,550152
3,MNLI,multinli_1.0/multinli_1.0_dev_matched.jsonl,10000
4,MNLI,multinli_1.0/multinli_1.0_dev_mismatched.jsonl,10000
5,MNLI,multinli_1.0/multinli_1.0_train.jsonl,392702
6,ANLI,anli_v1.0/R1/dev.jsonl,1000
7,ANLI,anli_v1.0/R1/test.jsonl,1000
8,ANLI,anli_v1.0/R1/train.jsonl,16946
9,ANLI,anli_v1.0/R2/dev.jsonl,1000


## Inspect matches

Look at a few of the sentence pairs from NLI datasets that matched sentence pairs in our Watson dataset

In [9]:
match_details = pd.DataFrame(matches)
if not match_details.empty:
    # Show a small readable sample from each matching dataset.
    for name, group in match_details.groupby("dataset", sort=False):
        print(name)
        display(group.drop_duplicates(["watson_file", "watson_id"]).head(3))
else:
    print("No matched pairs to display.")

MNLI
XNLI


,dataset,source_file,source_row,source_id,language,watson_file,watson_id,premise,hypothesis
0,MNLI,multinli_1.0/multinli_1.0_dev_matched.jsonl,1,63735n,en,train.csv,165f1529a7,The new rights are nice enough,Everyone really likes the newest benefits
1,MNLI,multinli_1.0/multinli_1.0_dev_matched.jsonl,2,91383c,en,train.csv,ddd02d184c,This site includes a list of all award winners and a searchable database of Government Executive articles.,The Government Executive articles housed on the website are not able to be searched.
2,MNLI,multinli_1.0/multinli_1.0_dev_matched.jsonl,3,755e,en,test.csv,bce10710ee,uh i don't know i i have mixed emotions about him uh sometimes i like him but at the same times i love to see somebody beat him,"I like him for the most part, but would still enjoy seeing someone beat him."


,dataset,source_file,source_row,source_id,language,watson_file,watson_id,premise,hypothesis
9815,XNLI,XNLI-1.0/xnli.dev.jsonl,10,25,ar,train.csv,01bb627264,وكنت بحالة حسنة، وكان ذلك!,بعد أن قلت نعم ، انتهى الأمر.
9816,XNLI,XNLI-1.0/xnli.dev.jsonl,22,58,ar,test.csv,4010405695,مايكل سانتو، من شركة فايروال وشركاه، في بوفالو، نيويورك، كانت تلك التي صنعتها، أه، آه، اخترعت منظم O2 العالي قبل أن يبنيوا النار على الموقد بشكل جيد.,عاش سانتو في نيويورك وعمل على منظم O2 العالي.
9817,XNLI,XNLI-1.0/xnli.dev.jsonl,42,105,ar,train.csv,e46ed25ce7,و هي حقا لم تفهم .,هو عرف تماما ماذا نحن كان نتحدث عنه.


## Locate the matches in the original splits

Just out of curiosity, let's see which split (training/dev/test) had the duplicate examples.

In [10]:
# Map the original filenames to their published split names.
source_split_names = {
    "multinli_1.0_dev_matched.jsonl": "dev_matched",
    "multinli_1.0_dev_mismatched.jsonl": "dev_mismatched",
    "multinli_1.0_train.jsonl": "train",
    "xnli.dev.jsonl": "dev",
    "xnli.test.jsonl": "test",
}
assert {r["dataset"] for r in acquisition if r["status"] == "loaded"} >= {"MNLI", "XNLI"}

# Retain every matching location; do not choose just one source per pair.
located_matches = match_details.copy()
located_matches["source_split"] = located_matches["source_file"].map(
    lambda name: source_split_names[Path(name).name]
)
source_locations = located_matches.drop_duplicates(
    ["dataset", "source_split", "watson_file", "watson_id"]
)

# Include scanned source splits with no matching Watson rows.
scanned_locations = pd.DataFrame(coverage)
scanned_locations = scanned_locations[scanned_locations["dataset"].isin(["MNLI", "XNLI"])].copy()
scanned_locations["source_split"] = scanned_locations["source_file"].map(
    lambda name: source_split_names[Path(name).name]
)
counts = source_locations.groupby(["dataset", "source_split"]).size().rename("Watson_rows_matched")
split_summary = scanned_locations[["dataset", "source_split", "rows"]].rename(
    columns={"rows": "source_rows_scanned"}
).merge(counts.reset_index(), on=["dataset", "source_split"], how="left")
split_summary["Watson_rows_matched"] = split_summary["Watson_rows_matched"].fillna(0).astype(int)
display(split_summary)

# Verify how many distinct Watson rows these locations account for.
located_row_count = len(source_locations.drop_duplicates(["watson_file", "watson_id"]))
assert located_row_count == len(all_matched)
print(f"Located {located_row_count:,} distinct Watson rows out of {len(watson):,}.")
multiple_locations = source_locations.groupby(["watson_file", "watson_id"]).size().gt(1).sum()
print(f"Watson rows found in more than one source dataset/split: {multiple_locations:,}.")

Located 17,315 distinct Watson rows out of 17,315.
Watson rows found in more than one source dataset/split: 0.


,dataset,source_split,source_rows_scanned,Watson_rows_matched
0,MNLI,dev_matched,10000,9815
1,MNLI,dev_mismatched,10000,0
2,MNLI,train,392702,0
3,XNLI,dev,37350,2490
4,XNLI,test,75150,5010
